# Step 1: Prepare the dataset

This notebook only prepares data. It does **not** train anything.

1. Load and clean the Wikipedia movie plots
2. Generate keywords for each plot (using few-shot prompting)
3. Export `review.csv` so you can review, fix, or remove rows
4. Load reviewed data and build `data/train.jsonl` and `data/valid.jsonl`

Then open `02_train.ipynb`.

## Settings

In [1]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

N_EXAMPLES = 3000
VALID_SIZE = 150
MIN_WORDS, MAX_WORDS = 100, 400
MIN_GENRE_COUNT = 100     # drop genres with fewer plots than this
MAX_SEQ_LEN = 768         # must match 02_train.ipynb

DATA_DIR = "data"
REVIEW_CSV = "review.csv"
SEED = 42

## 1. Load and clean

In [2]:
import pandas as pd
from datasets import load_dataset

ds = load_dataset("vishnupriyavr/wiki-movie-plots-with-summaries", split="train")
df = ds.to_pandas()
print(df.shape)
print(df.columns.tolist())
df.head(3)

/Users/darsh/movie-plots/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(34886, 9)
['Release Year', 'Title', 'Origin/Ethnicity', 'Director', 'Cast', 'Genre', 'Wiki Page', 'Plot', 'PlotSummary']


,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot,PlotSummary
0,1901,Kansas Saloon Smashers,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Kansas_Saloon_Sm...,"A bartender is working at a saloon, serving dr...",Carrie Nation and her followers burst into a s...
1,1901,Love by the Light of the Moon,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Love_by_the_Ligh...,"The moon, painted with a smiling face hangs ov...","The moon, painted with a smiling face hangs ov..."
2,1901,The Martyred Presidents,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/The_Martyred_Pre...,"The film, just over a minute long, is composed...","The film, just over a minute long, is composed..."


If the next cell gives a column error, look at the column names printed above and change `PLOT_COL` and `GENRE_COL`.

In [3]:
import re

PLOT_COL = "Plot"
GENRE_COL = "Genre"

df = df[[PLOT_COL, GENRE_COL]].dropna().copy()
df.columns = ["plot", "genre"]

df["genre"] = df["genre"].str.strip().str.lower()
df = df[~df["genre"].isin(["unknown", "", "nan"])]

def normalize_genre(g: str) -> str:
    g = str(g).lower().strip()
    g = re.sub(r"\bscience fiction\b", "sci-fi", g)
    parts = re.split(r"[,/;]|(?<!sci)-(?!fi)", g)
    clean_parts = sorted(list(set(p.strip() for p in parts if p.strip())))
    return ", ".join(clean_parts)

df["genre"] = df["genre"].apply(normalize_genre)

df["plot"] = df["plot"].str.replace(r"\s+", " ", regex=True).str.strip()
df["n_words"] = df["plot"].str.split().str.len()
df = df[(df["n_words"] >= MIN_WORDS) & (df["n_words"] <= MAX_WORDS)]

# Filter: keep movies where all constituent tags are well-represented (count inside combinations)
tag_counts = df["genre"].str.split(", ").explode().value_counts()
valid_genres = set(tag_counts[tag_counts >= MIN_GENRE_COUNT].index)
df = df[df["genre"].map(lambda g: all(t in valid_genres for t in g.split(", ")))]

df = df.sample(n=min(N_EXAMPLES, len(df)), random_state=SEED).reset_index(drop=True)
print("Using", len(df), "plots")

print("\n--- Individual genre tag occurrences (across all plots) ---")
print(df["genre"].str.split(", ").explode().value_counts())

print("\n--- Top genre combinations ---")
print(df["genre"].value_counts().head(15))

Using 3000 plots

--- Individual genre tag occurrences (across all plots) ---
genre
drama              1085
comedy              724
romance             274
action              236
western             151
thriller            136
crime               135
horror               99
musical              88
adventure            86
romantic comedy      65
crime drama          55
mystery              54
family               49
sci-fi               47
war                  46
film noir            44
Name: count, dtype: int64

--- Top genre combinations ---
genre
drama              901
comedy             578
romance            171
action             165
western            144
thriller           104
crime               88
horror              82
adventure           79
romantic comedy     64
musical             60
crime drama         55
comedy, drama       52
film noir           44
sci-fi              42
Name: count, dtype: int64


## 2. Generate keywords

The dataset has genres but no keywords, so the model reads each plot and extracts 3 to 5 keywords.
We use few-shot examples to prevent filler text, meta-labels, or prompt echoing.
Results are cached in a jsonl file, so you can stop and resume anytime.

In [4]:
import hashlib, json, os, re
from tqdm.auto import tqdm
from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler

kw_model, kw_tok = load(BASE_MODEL)
greedy = make_sampler(temp=0.0)

KW_FILE = f"keywords_cache_{BASE_MODEL.replace('/', '_')}.jsonl"
plot_key = lambda plot: hashlib.md5(plot.encode()).hexdigest()[:12]

cache = {}
if os.path.exists(KW_FILE):
    with open(KW_FILE) as f:
        for line in f:
            r = json.loads(line)
            cache[r["id"]] = r["keywords"]

FEW_SHOT_MESSAGES = [
    {
        "role": "system",
        "content": "Extract 3 to 5 concise plot keywords (comma-separated). Mention specific themes, key objects, settings, or roles. Do NOT use character proper names. Output ONLY comma-separated keywords with no introductory text or labels."
    },
    {
        "role": "user",
        "content": "Movie Plot: A group of astronauts travel through a wormhole near Saturn in search of a new habitable planet for humanity as Earth faces environmental collapse.\n\nKeywords:"
    },
    {
        "role": "assistant",
        "content": "space exploration, wormhole, environmental collapse, survival, new planet"
    },
    {
        "role": "user",
        "content": "Movie Plot: In 1947, banker Andy Dufresne is convicted of murder and sent to Shawshank prison. He befriends inmate Red and helps corrupt Warden Norton with money laundering while quietly digging an escape tunnel.\n\nKeywords:"
    },
    {
        "role": "assistant",
        "content": "wrongful conviction, prison life, money laundering, escape tunnel, friendship"
    },
]

FORBIDDEN = {"theme", "themes", "setting", "key object", "character", "situation", "movie", "film", "plot"}

def clean_keywords(text):
    first_line = re.sub(r"[*_#]", "", text.strip().split("\n")[0])
    parts = [p.strip().strip(".\"'*-").lower() for p in first_line.split(",")]
    cleaned = []
    for p in parts:
        if not p or len(p.split()) > 4:
            continue
        if p in FORBIDDEN or p.startswith("themes:") or p.startswith("keywords:"):
            continue
        cleaned.append(p)
    return cleaned[:5]

def get_keywords(plot):
    short = " ".join(plot.split()[:250])
    msgs = FEW_SHOT_MESSAGES + [
        {"role": "user", "content": f"Movie Plot: {short}\n\nKeywords:"}
    ]
    prompt = kw_tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    response = generate(kw_model, kw_tok, prompt=prompt, max_tokens=30, sampler=greedy)
    return clean_keywords(response)

with open(KW_FILE, "a") as f:
    for plot in tqdm(df["plot"]):
        k = plot_key(plot)
        if k in cache:
            continue
        cache[k] = get_keywords(plot)
        f.write(json.dumps({"id": k, "keywords": cache[k]}) + "\n")
        f.flush()

df["id"] = df["plot"].map(plot_key)
df["keywords"] = df["id"].map(cache)
df = df[df["keywords"].str.len() >= 2].reset_index(drop=True)
print("Plots with usable keywords:", len(df))

100%|██████████████████████████████████████████████████████████████████| 3000/3000 [40:42<00:00,  1.23it/s]

Plots with usable keywords: 3000


## 3. Export for review, then fix by hand

The next cell writes `review.csv`. Open it in Excel, Numbers, Google Sheets, or a CSV editor:
- Set **keep** to `0` to remove a row.
- Edit the **genre** or **keywords** cells if you want to tweak them (keywords are separated by commas).

Save it with the same name. **The CSV is the source of truth from here on.**
Delete or rename `review.csv` if you want to regenerate it from scratch.

In [5]:
if os.path.exists(REVIEW_CSV):
    print(f"{REVIEW_CSV} already exists, not overwriting it. Delete or rename it if you want to regenerate.")
else:
    out = df[["id", "genre", "keywords", "plot"]].copy()
    out["keywords"] = out["keywords"].map(", ".join)
    out["keep"] = 1
    out.to_csv(REVIEW_CSV, index=False)
    print(f"Wrote {REVIEW_CSV} with {len(out)} rows for review.")

Wrote review.csv with 3000 rows for review.


## 4. Load your reviewed data and check lengths

In [6]:
final = pd.read_csv(REVIEW_CSV)
final = final[final["keep"] == 1].copy()

# Ensure genre and keywords are clean strings and valid
final["genre"] = final["genre"].fillna("").astype(str).str.strip()
final["keywords"] = final["keywords"].fillna("").astype(str).map(
    lambda s: [k.strip() for k in s.split(",") if k.strip()]
)
final = final[(final["genre"] != "") & (final["keywords"].str.len() >= 2)].reset_index(drop=True)
print("Rows after your review:", len(final))

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(BASE_MODEL)

def n_tokens(r):
    msgs = [
        {"role": "user", "content": f"Genre: {r.genre}. Keywords: {', '.join(r.keywords)}"},
        {"role": "assistant", "content": r.plot},
    ]
    text = tok.apply_chat_template(msgs, tokenize=False)
    return len(tok(text).input_ids)

final["n_tokens"] = [n_tokens(r) for r in final.itertuples()]
print(final["n_tokens"].describe())
print("Examples longer than MAX_SEQ_LEN (they will be cut off):", (final["n_tokens"] > MAX_SEQ_LEN).sum())

Rows after your review: 3000
count    3000.000000
mean      341.540667
std       111.594260
min       159.000000
25%       243.750000
50%       325.500000
75%       427.000000
max       642.000000
Name: n_tokens, dtype: float64
Examples longer than MAX_SEQ_LEN (they will be cut off): 0


## 5. Build the training files

In [7]:
import random

os.makedirs(DATA_DIR, exist_ok=True)

examples = [
    {"messages": [
        {"role": "user", "content": f"Genre: {r.genre}. Keywords: {', '.join(r.keywords)}"},
        {"role": "assistant", "content": r.plot},
    ]}
    for r in final.itertuples() if r.n_tokens <= MAX_SEQ_LEN
]
random.Random(SEED).shuffle(examples)

valid, train = examples[:VALID_SIZE], examples[VALID_SIZE:]

def save(path, rows):
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

save(f"{DATA_DIR}/train.jsonl", train)
save(f"{DATA_DIR}/valid.jsonl", valid)
save(f"{DATA_DIR}/test.jsonl", valid[:20])

print("train:", len(train), " valid:", len(valid))
print(json.dumps(train[0], indent=2)[:900])

train: 2850  valid: 150
{
  "messages": [
    {
      "role": "user",
      "content": "Genre: family. Keywords: shipwreck, rescue, adventure, peril, shipwrecked father"
    },
    {
      "role": "assistant",
      "content": "In company with her younger brother Robert (Keith Hamshere), and her elderly yet young-at-heart friend Professor Jacques Paganel (Maurice Chevalier), teenage Mary Grant (Hayley Mills) journeys to Glasgow, Scotland, to persuade the comically brave Lord Glenarvan (Wilfrid Hyde-White) to rescue her shipwrecked father, Captain John Grant (Jack Gwillim). The expedition sets sail and ventures halfway around the world, to both South America and New Zealand, respectively. The party suffers many assorted perils including an earthquake, a flood, a fire, an attack by a giant condor and an erupting volcano before finding and rescuing Captain Grant. A subplot involves a gunrunner named Thomas Ayerton


Done. Next: open `02_train.ipynb`.